# Ward census walkthrough

Hourly occupancy for six wards. The only real number that comes back is a midnight headcount.
Run from the `ward-twin-synthetic` folder so `src/` imports resolve.

See also `docs/METHOD.md`.

In [ ]:
import sys
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
from src.simulate import make_ward_data, WARD_NAMES
from src.glm import design, fit_nb_glm, nb_interval, pit_values

rng = np.random.default_rng(7)
df = make_ward_data(rng)
df.groupby('ward_name')['beds'].agg(['mean','std']).round(1)

Fit the NB mean on the first 90 days. Coverage and the PIT histogram are the honesty checks — MAE is the scoreboard.

In [ ]:
train = df[df['t'] < 90 * 24]
test = df[df['t'] >= 90 * 24].copy()
beta, alpha = fit_nb_glm(design(train), train['beds'].values.astype(float))
test['pred'] = np.exp(design(test) @ beta)
test['lo'], test['hi'] = nb_interval(test['pred'].values, alpha)
mae = np.abs(test['beds'] - test['pred']).mean()
cover = ((test['beds'] >= test['lo']) & (test['beds'] <= test['hi'])).mean()
pit = pit_values(test['beds'].values, test['pred'].values, alpha)
print(f'MAE {mae:.2f}  coverage {cover:.0%}  alpha {alpha:.4f}')
print('PIT deciles', np.histogram(pit, bins=10, range=(0,1))[0])

If two midnights miss in a row, the gate is the story — not the MAE. Then open `site/index.html`.

In [ ]:
import main as m
m.main()
print('see outputs/dashboard.json and site/index.html')